In [2]:
%pip install nltk

  Using cached nltk-3.10.4-py3-none-any.whl.metadata (5.0 kB)
Using cached nltk-3.10.4-py3-none-any.whl (1.7 MB)

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from finnews_sentiment.features.preprocess import preprocess_text, prepare_dataset, add_basic_features

In [4]:
train_df = pd.read_csv("data/ag_news_train.csv")
test_df = pd.read_csv("data/ag_news_test.csv")
print(train_df.shape, test_df.shape)

(120000, 3) (7600, 3)


In [5]:
for t in train_df["text"].sample(5, random_state=1):
    print("ДО:   ", t[:200])
    print("ПОСЛЕ:", preprocess_text(t)[:200])
    print()

ДО:    BEA buzzing about Beehive BURLINGAME, CALIF.-- BEA Systems is expanding the open source Beehive initiative but still has no plans to participate in the Eclipse open source tools organization, despite 
ПОСЛЕ: bea buzzing beehive burlingame calif bea system expanding open source beehive initiative still plan participate eclipse open source tool organization despite embrace beehive eclipse

ДО:    Saudi Violence Pushes Oil Prices Higher Oil prices were forced higher today by an outbreak of violence in Saudi Arabia and concerns that Opec may cut output. The cost of a barrel of light, sweet crude
ПОСЛЕ: saudi violence push oil price higher oil price forced higher today outbreak violence saudi arabia concern opec may cut output cost barrel light sweet crude oil climbed

ДО:    Storm shows Haiti #39;s straits Muddy waters still occupied the heart of Gonaives last weekend, three weeks after Tropical Storm Jeanne provoked flash floods that killed some 1,900 and left 900 presum
ПОСЛЕ: sto

In [6]:
for t in ["America #39;s top diplomat", "Dow Jones  amp; Co", "a  quot;strong dollar, quot;", "", None]:
    print(repr(t), "->", repr(preprocess_text(t)))

'America #39;s top diplomat' -> 'america top diplomat'
'Dow Jones  amp; Co' -> 'dow jones'
'a  quot;strong dollar, quot;' -> 'strong dollar'
'' -> ''
None -> ''


In [7]:
train_clean = prepare_dataset(train_df)
test_clean = prepare_dataset(test_df)

print("train:", len(train_df), "->", len(train_clean))
print("test:", len(test_df), "->", len(test_clean))

train: 120000 -> 119050
test: 7600 -> 7599


In [8]:
print(train_clean["label_name"].value_counts())
train_clean[["text", "text_clean", "label", "label_name"]].head()

label_name
World       29835
Sports      29828
Business    29792
Sci/Tech    29595
Name: count, dtype: int64


,text,text_clean,label,label_name
0,Wall St. Bears Claw Back Into the Black (Reute...,wall bear claw back black reuters reuters shor...,2,Business
1,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle look toward commercial aerospace reute...,2,Business
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil economy cloud stock outlook reuters reuter...,2,Business
3,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halt oil export main southern pipeline re...,2,Business
4,"Oil prices soar to all-time record, posing new...",oil price soar alltime record posing new menac...,2,Business


In [10]:
train_clean = add_basic_features(train_clean)
test_clean = add_basic_features(test_clean)
train_clean[["text_len_words", "number_count"]].describe()

,text_len_words,number_count
count,119050.000000,119050.000000
mean,24.105670,1.356077
std,6.220827,1.742677
min,6.000000,0.000000
25%,20.000000,0.000000
50%,24.000000,1.000000
75%,27.000000,2.000000
max,98.000000,56.000000


In [9]:
train_clean.to_csv("data/train_clean.csv", index=False)
test_clean.to_csv("data/test_clean.csv", index=False)

## Выводы

- предобработка вынесена в src/finnews_sentiment/features/preprocess.py, в тетрадке только импорт
- чистим артефакты AG News (#39; quot; amp;), убираем цифры, пунктуацию, стоп-слова, лемматизируем
- шаг с тикерами убрала: после lower() он ничего не находил, а для AG News не нужен
- удалено ... строк из train и ... из test (пустые, короткие, дубли)
- баланс классов после очистки ...
- метки: label 0-3 и label_name (World, Sports, Business, Sci/Tech)